In [15]:
import pandas as pd
import numpy as np
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [16]:
df = pd.read_csv("../data/processed/cases.csv")

df["text_full"] = df["text_full"].fillna("").astype(str)

df.head()

,case_id,nomor_perkara,jenis_perkara,ringkasan_fakta,panjang_teks,text_full,amar_putusan
0,case_001,052/phpu.c.1-ii/2004 demi keadilan berdasarkan...,Pertanahan,p u t u s a n perkara nomor: 052/phpu.c.1-ii/2...,1000,p u t u s a n perkara nomor: 052/phpu.c.1-ii/2...,"mengadili, dan memutus perkara konstitusi pada..."
1,case_002,95/puu-xii/2014 demi keadilan berdasarkan ketu...,Pertanahan,salinan putusan nomor 95/puu-xii/2014 demi kea...,60565,salinan putusan nomor 95/puu-xii/2014 demi kea...,amar putusan mengadili; menyatakan
2,case_003,96/puu-xiv/2016 demi keadilan berdasarkan ketu...,Pertanahan,salinan putusan nomor 96/puu-xiv/2016 demi kea...,91722,salinan putusan nomor 96/puu-xiv/2016 demi kea...,amar putusan sebagai berikut: 1) menyatakan
3,case_004,113-10-18/phpu.dpr-dprd/xvii/2019 demi keadila...,Pertanahan,sal inan ketetapan nomor 113-10-18/phpu.dpr-dp...,912,sal inan ketetapan nomor 113-10-18/phpu.dpr-dp...,mengadili perkara konstitusi pada tingkat pert...
4,case_005,99/puu-xx/2022 demi keadilan berdasarkan ketuh...,Pertanahan,1 s ali n an ketetapan nomor 99/puu-xx/2022 de...,869,1 s ali n an ketetapan nomor 99/puu-xx/2022 de...,memutuskan mencabut permohonan a quo; d. bahwa...


In [17]:
vectorizer = TfidfVectorizer(max_features=5000)

tfidf_matrix = vectorizer.fit_transform(df["text_full"])

print(tfidf_matrix.shape)

(31, 5000)


In [18]:
def retrieve(query, k=5):

    query_vec = vectorizer.transform([query])

    sim = cosine_similarity(query_vec, tfidf_matrix).flatten()

    top_idx = sim.argsort()[-k:][::-1]

    results = []

    for i in top_idx:
        results.append({
            "case_id": df.iloc[i]["case_id"],
            "similarity": float(sim[i])
        })

    return pd.DataFrame(results)

In [19]:
retrieve("sengketa tanah tanpa izin", k=5)

,case_id,similarity
0,case_003,0.195781
1,case_016,0.161672
2,case_013,0.118814
3,case_011,0.100828
4,case_012,0.098552


In [20]:
import re

def extract_amar(text):

    text = str(text).lower()

    pattern = r"(amar\s+putusan.*?)(menyatakan|memutuskan|demikian|ditetapkan)"

    match = re.search(pattern, text, re.DOTALL)

    if match:
        return match.group(1)[:2000]

    return "TIDAK DITEMUKAN"


if "amar_putusan" not in df.columns:
    df["amar_putusan"] = df["text_full"].apply(extract_amar)

In [21]:
case_solutions = dict(zip(df["case_id"], df["amar_putusan"]))

In [22]:
def predict_weighted(query, k=5):

    top_cases = retrieve(query, k)

    scores = {}

    for _, row in top_cases.iterrows():

        cid = row["case_id"]
        sim = row["similarity"]

        sol = case_solutions[cid]

        scores[sol] = scores.get(sol, 0) + sim

    return max(scores, key=scores.get)

In [23]:
queries = [
    {"query": "sengketa tanah tanpa izin", "ground_truth": "case_001"},
    {"query": "pengujian undang undang kehutanan", "ground_truth": "case_002"},
    {"query": "perselisihan hasil pemilu", "ground_truth": "case_010"},
    {"query": "larangan pemakaian tanah", "ground_truth": "case_003"},
    {"query": "peraturan cipta kerja", "ground_truth": "case_006"}
]

In [24]:
y_true = []
y_pred = []

for q in queries:

    result = retrieve(q["query"], k=1)

    pred_case = result.iloc[0]["case_id"]

    y_true.append(q["ground_truth"])
    y_pred.append(pred_case)

In [25]:
retrieval_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
    "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    "f1_score": f1_score(y_true, y_pred, average="macro", zero_division=0)
}

retrieval_metrics

{'accuracy': 0.6,
 'precision': 0.4166666666666667,
 'recall': 0.5,
 'f1_score': 0.4444444444444444}

In [26]:
y_true = []
y_pred = []

for q in queries:

    true_sol = df[df["case_id"] == q["ground_truth"]]["amar_putusan"].values[0]

    pred_sol = predict_weighted(q["query"])

    y_true.append(true_sol)
    y_pred.append(pred_sol)

In [27]:
prediction_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
    "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    "f1_score": f1_score(y_true, y_pred, average="macro", zero_division=0)
}

prediction_metrics

{'accuracy': 0.4,
 'precision': 0.25,
 'recall': 0.3333333333333333,
 'f1_score': 0.27777777777777773}

In [28]:
os.makedirs("../data/eval", exist_ok=True)

pd.DataFrame([retrieval_metrics]).to_csv(
    "../data/eval/retrieval_metrics.csv",
    index=False
)

pd.DataFrame([prediction_metrics]).to_csv(
    "../data/eval/prediction_metrics.csv",
    index=False
)

print("SEMUA HASIL TERSIMPAN")

SEMUA HASIL TERSIMPAN
